In [29]:
import copy 
import random 
import matplotlib.pyplot as plt 
import numpy as np 
import pandas as pd 
import torch 
import torch.nn as nn

SEED = 7 
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)


In [30]:
torch.set_num_threads(1)

In [31]:
def create_task(axis,total_samples=500):
       x  = torch.randn(total_samples ,2 )
       y=(x[:,axis ]>0 ).long()
       x_train=x[:400]
       y_train=y[:400]
       x_test=x[:400]
       y_test=y[:400]
       return x_train,y_train,x_test,y_test 

x1_train,y1_train,x1_test,y1_test=create_task(axis=0)
x2_train,y2_train,x2_test,y2_test=create_task(axis=1)
print(x1_train.shape,y1_train.shape,x1_test.shape,y1_test.shape)
print(x2_train.shape,y2_train.shape,x2_test.shape,y2_test.shape)

torch.Size([400, 2]) torch.Size([400]) torch.Size([400, 2]) torch.Size([400])
torch.Size([400, 2]) torch.Size([400]) torch.Size([400, 2]) torch.Size([400])


In [32]:
class continualNet(nn.Module):
       def __init__(self):
              super().__init__()
              self.network=nn.Sequential(
                     nn.Linear(2,16),
                     nn.ReLU(),
                     nn.Linear(16,2)
              )
       def forward(self,x):
              return self.network(x)
template_model=continualNet()
initial_weights=copy.deepcopy(template_model.state_dict())

def create_fresh_model():
       model=continualNet()
       model.load_state_dict(initial_weights)
       return model

In [33]:
## accuracy function 
def calculate_accuracy(model,x,y):
       model.eval()
       with torch.no_grad():
              outputs=model(x)
              predictions=outputs.argmax(dim=1)
              correct=(predictions==y).float()
              accuracy=correct.mean()
       return accuracy.item()*100

In [34]:
#general training function
def train_model(model,X,y,epochs=50,learning_rate=0.05,regularization_function=None,method_name='Model'):
       optimizer=torch.optim.SGD(model.parameters(),lr=learning_rate)
       criterion=nn.CrossEntropyLoss()
       model.train()

       for epoch in range(1,epochs+1):
              optimizer.zero_grad()
              outputs=model(X)
              task_loss=criterion(outputs,y)
              total_loss= task_loss 
              total_loss.backward()
              optimizer.step()

              if epoch in [1,25,50]:
                     acc=calculate_accuracy(model,X,y)
                     print(f"{method_name} Epoch {epoch}: Loss={total_loss.item():.4f}, Accuracy={acc:.2f}%")

In [35]:
print("\n"+"="*60)
print("Baseline Training:")
print("="*60)

print("\n Training the baseline task1 ")
baseline_model=create_fresh_model()
print('\n print baseline model for task 1 ')
train_model(model=baseline_model,X=x1_train,y=y1_train,epochs=50,learning_rate=0.05,method_name='Baseline Task1')
baseline_task1_before=calculate_accuracy(baseline_model,x1_test,y1_test)
print(f"\n Task1 accuracy before task2 :" f"{baseline_task1_before:.2f}%")

print("\n Training the baseline task2 ")
baseline_model=create_fresh_model()
print('\n print baseline model for task 2 ')
train_model(model=baseline_model,X=x2_train,y=y2_train,epochs=50,learning_rate=0.05,method_name='Baseline Task2')
baseline_task2_before=calculate_accuracy(baseline_model,x2_test,y2_test)
print(f"\n Task2 accuracy before task3 :" f"{baseline_task2_before:.2f}%")

baseline_task1_after=calculate_accuracy(baseline_model,x1_test,y1_test)
baseline_task2_after=calculate_accuracy(baseline_model,x2_test,y2_test)
print('\n Baseline final results:')
print(f"Task1 accuracy after training: {baseline_task1_after:.2f}%")
print(f"Task2 accuracy after training: {baseline_task2_after:.2f}%")


Baseline Training:

 Training the baseline task1 

 print baseline model for task 1 
Baseline Task1 Epoch 1: Loss=0.7879, Accuracy=34.25%
Baseline Task1 Epoch 25: Loss=0.4838, Accuracy=95.75%
Baseline Task1 Epoch 50: Loss=0.3477, Accuracy=96.50%

 Task1 accuracy before task2 :96.50%

 Training the baseline task2 

 print baseline model for task 2 
Baseline Task2 Epoch 1: Loss=0.8083, Accuracy=28.25%
Baseline Task2 Epoch 25: Loss=0.5513, Accuracy=78.00%
Baseline Task2 Epoch 50: Loss=0.4410, Accuracy=90.75%

 Task2 accuracy before task3 :90.75%

 Baseline final results:
Task1 accuracy after training: 53.50%
Task2 accuracy after training: 90.75%


In [36]:
# ============================================================
# 6. Baseline training
# ============================================================

print("\n" + "=" * 60)
print("BASELINE TRAINING")
print("=" * 60)


baseline_model = create_fresh_model()


# ---------------- Task 1 ----------------

print("\nTraining Baseline on Task 1")

train_model(
    model=baseline_model,
    X=x1_train,
    y=y1_train,
    method_name="Baseline Task 1"
)


baseline_task1_before = calculate_accuracy(
    baseline_model,
    x1_test,
    y1_test
)


print(
    f"\nTask 1 accuracy before Task 2: "
    f"{baseline_task1_before:.2f}%"
)


# ---------------- Task 2 ----------------

print("\nTraining Baseline on Task 2")

train_model(
    model=baseline_model,
    X=x2_train,
    y=y2_train,
    method_name="Baseline Task 2"
)


baseline_task1_after = calculate_accuracy(
    baseline_model,
    x1_test,
    y1_test
)

baseline_task2_after = calculate_accuracy(
    baseline_model,
    x2_test,
    y2_test
)


print("\nBaseline final result")

print(
    f"Task 1 accuracy after Task 2: "
    f"{baseline_task1_after:.2f}%"
)

print(
    f"Task 2 accuracy: "
    f"{baseline_task2_after:.2f}%"
)


BASELINE TRAINING

Training Baseline on Task 1
Baseline Task 1 Epoch 1: Loss=0.7879, Accuracy=34.25%
Baseline Task 1 Epoch 25: Loss=0.4838, Accuracy=95.75%
Baseline Task 1 Epoch 50: Loss=0.3477, Accuracy=96.50%

Task 1 accuracy before Task 2: 96.50%

Training Baseline on Task 2
Baseline Task 2 Epoch 1: Loss=0.8876, Accuracy=55.25%
Baseline Task 2 Epoch 25: Loss=0.5515, Accuracy=75.00%
Baseline Task 2 Epoch 50: Loss=0.4114, Accuracy=83.50%

Baseline final result
Task 1 accuracy after Task 2: 61.75%
Task 2 accuracy: 83.50%
